In [1]:
import os
import re
import json
import shutil
import hashlib
import subprocess
import requests
import urllib.parse
from datetime import datetime, timedelta
from typing import TypedDict, List, Dict, Any
import pandas as pd
from bs4 import BeautifulSoup

from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from exa_py import Exa
from composio import Composio
from dotenv import load_dotenv
load_dotenv()  # This loads your variables from the local .env file

# ==========================================
# 1. API KEYS & CREDENTIALS SETUP
# ==========================================
os.environ ["EXA_API_KEY"] = os.getenv("EXA_API_KEY", "")
os.environ ["COMPOSIO_API_KEY"] = os.getenv("COMPOSIO_API_KEY", "")
os.environ ["GETEMAIL_API_KEY"] = os.getenv("GETEMAIL_API_KEY", "")

# Initialize Search Clients
composio = Composio(api_key=os.environ["COMPOSIO_API_KEY"])
exa = Exa(api_key=os.environ["EXA_API_KEY"])

# Local LLM for Private CV Ingestion & Fallback
ollama_llm = ChatOllama(model="qwen2.5:7b", base_url="http://localhost:11434")
ollama_embeddings = OllamaEmbeddings(model="nomic-embed-text", base_url="http://localhost:11434")

# Directories & Storage
RESUMES_DIR = r"C:\Users\simar\Desktop\CV_2026"
CACHE_FILE = r"C:\Users\simar\Downloads\multi_resume_cache.json"
CHROMA_PERSIST_DIR = r"C:\Users\simar\Downloads\chroma_db_resumes"
EXCEL_PATH = r"C:\Users\simar\Downloads\Matched_Jobs.xlsx"

# Path to authenticated Claude executable
CLAUDE_EXE = r"C:\Users\simar\.local\bin\claude.exe"

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def get_file_hash(filepath: str) -> str:
    hasher = hashlib.md5()
    with open(filepath, 'rb') as f:
        buf = f.read(65536)
        while len(buf) > 0:
            hasher.update(buf)
            buf = f.read(65536)
    return hasher.hexdigest()

def load_cache() -> Dict[str, Any]:
    if os.path.exists(CACHE_FILE):
        try:
            with open(CACHE_FILE, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return {}
    return {}

def save_cache(cache_data: Dict[str, Any]):
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache_data, f, indent=2)

def parse_duration_to_days(user_input: str) -> int:
    txt = user_input.lower().strip()
    if "24" in txt or "hour" in txt or "today" in txt or "1 day" in txt:
        return 1
    day_match = re.search(r"(\d+)\s*day", txt)
    if day_match:
        return int(day_match.group(1))
    if "week" in txt:
        week_match = re.search(r"(\d+)\s*week", txt)
        return int(week_match.group(1)) * 7 if week_match else 7
    if "month" in txt:
        month_match = re.search(r"(\d+)\s*month", txt)
        return int(month_match.group(1)) * 30 if month_match else 30
    return 7

# ==========================================
# 3. CHROMA SETUP
# ==========================================
vectorstore = Chroma(
    persist_directory=CHROMA_PERSIST_DIR,
    embedding_function=ollama_embeddings
)

def sync_resumes_to_chroma(resumes_dir: str):
    if not os.path.exists(resumes_dir):
        return
    cache = load_cache()
    pdf_files = [f for f in os.listdir(resumes_dir) if f.endswith(".pdf")]

    for file_name in pdf_files:
        file_path = os.path.join(resumes_dir, file_name)
        file_hash = get_file_hash(file_path)
        persona_key = os.path.splitext(file_name)[0]

        if persona_key in cache and cache[persona_key].get("pdf_hash") == file_hash:
            continue

        print(f"📦 Indexing updated resume persona: {file_name} into ChromaDB...")
        loader = PyPDFLoader(file_path)
        docs = loader.load_and_split()
        for d in docs:
            d.metadata["persona_key"] = persona_key
            d.metadata["file_name"] = file_name

        vectorstore.add_documents(docs)
        print(f"  ✓ {file_name} indexed with persona metadata.")

sync_resumes_to_chroma(RESUMES_DIR)

# ==========================================
# 4. PIPELINE STATE
# ==========================================
class PipelineState(TypedDict):
    candidate_name: str
    roles: List[str]
    target_geos: List[str]
    job_duration_days: int
    duration_label: str
    target_approved_count: int
    resume_summary: str
    raw_jobs: List[Dict[str, Any]]
    evaluated_jobs: List[Dict[str, Any]]
    approved_jobs: List[Dict[str, Any]]
    low_match_summary: str
    search_iteration: int
    user_decision: str
    search_mode: str                 # "both" (default, parallel) | "linkedin" | "career" (= web agent)
    target_domain: str               # blank = derive from CV
    career_company_count: int
    role_variants: List[str]
    shortlisted_companies: List[Dict[str, Any]]
    company_batch: List[Dict[str, Any]]
    career_jobs: List[Dict[str, Any]]
    board_jobs: List[Dict[str, Any]]
    vc_funds: List[str]              # blank = Accel, Matrix Partners + local funds
    fund_boards: List[Dict[str, Any]]
    new_fund_boards: List[Dict[str, Any]]

# ==========================================
# 5. GRAPH NODES
# ==========================================

# Node 1: Multi-Resume Cached Reader
def parse_resume_node(state: PipelineState) -> Dict[str, Any]:
    persona_key = state.get("candidate_name", "Default")
    print(f"\n📄 Node 1: Checking Resume Profile Cache for '{persona_key}'...")

    cache = load_cache()
    expected_file = os.path.join(RESUMES_DIR, f"{persona_key}.pdf")
    current_hash = get_file_hash(expected_file) if os.path.exists(expected_file) else None

    if persona_key in cache:
        entry = cache[persona_key]
        if current_hash is None or entry.get("pdf_hash") == current_hash:
            print(f"  ⚡ [Instant] Loaded cached profile for '{persona_key}' (0ms)!")
            return {"resume_summary": entry["summary"]}

    print(f"  ⚙️ Generating new summary for '{persona_key}' via local Ollama...")
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3, "filter": {"persona_key": persona_key}})
    docs = retriever.invoke(f"Key technical skills, target roles, impact for {persona_key}")
    context = "\n\n".join([d.page_content for d in docs])

    if not context.strip() and os.path.exists(expected_file):
        loader = PyPDFLoader(expected_file)
        context = "\n\n".join([d.page_content for d in loader.load()])

    prompt = f"""Summarize this candidate profile into concise bullet points for job matching:
1. Primary Roles (e.g. Chief of Staff, Product Manager, AI Product Builder)
2. Core Technical, AI & Product Competencies
3. Industry Domains & Impact

Resume Context:
{context}"""

    resp = ollama_llm.invoke(prompt)
    summary_text = resp.content

    cache[persona_key] = {
        "file_name": f"{persona_key}.pdf",
        "pdf_hash": current_hash,
        "last_updated": datetime.now().isoformat(),
        "summary": summary_text
    }
    save_cache(cache)
    print(f"  ✓ Summary cached successfully for '{persona_key}'!")
    return {"resume_summary": summary_text}


# Fast Live LinkedIn Direct Guest Scraper
def fetch_live_linkedin_jobs(role: str, geo: str, days: int = 3, limit: int = 20) -> List[Dict[str, Any]]:
    seconds = days * 86400
    region, remote = parse_geo(geo)  # 'European Union (Remote)' -> region + LinkedIn remote filter
    encoded_role = urllib.parse.quote(role)
    encoded_geo = urllib.parse.quote(region or "Worldwide")
    url = f"https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search?keywords={encoded_role}&location={encoded_geo}&f_TPR=r{seconds}&start=0"
    if remote:
        url += "&f_WT=2"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36"}
    
    jobs = []
    try:
        resp = requests.get(url, headers=headers, timeout=8)
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, "html.parser")
            for post in soup.find_all("li")[:limit]:
                title_tag = post.find("h3", class_="base-search-card__title")
                company_tag = post.find("h4", class_="base-search-card__subtitle")
                location_tag = post.find("span", class_="job-search-card__location")
                link_tag = post.find("a", class_="base-card__full-link")
                date_tag = post.find("time")
                
                if title_tag and link_tag:
                    jobs.append({
                        "Job Profile": title_tag.get_text(strip=True),
                        "Target Company": company_tag.get_text(strip=True) if company_tag else "Unknown Company",
                        "Location": location_tag.get_text(strip=True) if location_tag else geo,
                        "Job Link": link_tag.get("href", "").split("?")[0],
                        "Platform Name": "LinkedIn",
                        "Date Posted": date_tag.get_text(strip=True) if date_tag else f"Past {days} days"
                    })
    except Exception as e:
        print(f"  ⚠️ Live LinkedIn query issue for {role}: {e}")
    return jobs


# Node 2: Fast Dual-Engine Job Search
def search_jobs_node(state: PipelineState) -> Dict[str, Any]:
    days = state.get("job_duration_days", 3)
    iteration = state.get("search_iteration", 1)
    if state.get("search_mode", "both") == "career":
        return {"raw_jobs": [], "search_iteration": iteration + 1}
    print(f"\n🔍 Node 2: Searching live postings [Iteration {iteration}] (past {days} days)...")

    all_jobs = []

    for role in state["roles"]:
        for geo in state["target_geos"]:
            print(f"  ⚡ Querying live LinkedIn index for '{role}' in '{geo}'...")
            li_jobs = fetch_live_linkedin_jobs(role, geo, days=days, limit=15)
            all_jobs.extend(li_jobs)

    # ATS boards (Greenhouse/Lever/Ashby...) are searched by the web agent (Node 2c)

    unique_map = {j["Job Link"]: j for j in all_jobs if j.get("Job Link")}
    final_raw = list(unique_map.values())

    print(f"  ✓ Found {len(final_raw)} fresh live postings in ~3 seconds!")
    return {"raw_jobs": final_raw, "search_iteration": iteration + 1}


# ==========================================
# CAREER-PAGE SEARCH (runs in parallel with the LinkedIn search)
# 1. Role-first: find postings for the role in the target area and collect the hiring companies
# 2. Domain: add companies in the user's domain headquartered in / with offices in the area
# 3. Read every shortlisted company's careers page / ATS board for matching roles
# ==========================================
from concurrent.futures import ThreadPoolExecutor

CAREER_HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36"}
CAREER_PATHS = ["/careers", "/jobs", "/careers/jobs", "/join-us", "/company/careers", "/about/careers"]
ATS_DOMAINS = ["greenhouse.io", "lever.co", "ashbyhq.com", "smartrecruiters.com", "workable.com", "myworkdayjobs.com"]
JOB_BOARD_DOMAINS = ["linkedin.com", "indeed.com", "glassdoor.com", "naukri.com", "wellfound.com", "builtin.com",
                     "monster.com", "ziprecruiter.com", "remoteok.com", "weworkremotely.com", "remote.co", "instahyre.com",
                     "iimjobs.com", "foundit.in", "otta.com", "welcometothejungle.com", "stepstone.de", "jooble.org"]
SKIP_COMPANY_DOMAINS = JOB_BOARD_DOMAINS + ["wikipedia.", "crunchbase.", "tracxn.", "ambitionbox.", "medium.",
                                            "youtube.", "forbes.", "reddit.", "quora.", "github."]

EU_PLACES = ["eu", "europe", "european union", "emea", "austria", "belgium", "bulgaria", "croatia", "cyprus", "czech",
             "czechia", "denmark", "estonia", "finland", "france", "germany", "greece", "hungary", "ireland", "italy",
             "latvia", "lithuania", "luxembourg", "malta", "netherlands", "poland", "portugal", "romania", "slovakia",
             "slovenia", "spain", "sweden", "berlin", "munich", "hamburg", "paris", "amsterdam", "rotterdam", "madrid",
             "barcelona", "dublin", "lisbon", "porto", "stockholm", "copenhagen", "helsinki", "warsaw", "krakow",
             "prague", "vienna", "brussels", "milan", "rome", "athens", "budapest", "bucharest", "tallinn", "vilnius",
             "riga", "sofia", "zagreb", "valencia"]
LOCATION_ALIASES = {
    "eu": EU_PLACES, "european union": EU_PLACES,
    "europe": EU_PLACES + ["uk", "united kingdom", "london", "switzerland", "zurich", "norway", "oslo"],
    "bangalore": ["bangalore", "bengaluru"], "bengaluru": ["bangalore", "bengaluru"],
    "gurgaon": ["gurgaon", "gurugram"], "gurugram": ["gurgaon", "gurugram"],
    "delhi": ["delhi", "ncr", "noida", "gurgaon", "gurugram"],
    "mumbai": ["mumbai", "bombay"],
    "london": ["london", "united kingdom", "uk"], "uk": ["uk", "united kingdom", "london", "england"],
    "netherlands": ["netherlands", "amsterdam", "rotterdam", "utrecht", "eindhoven"],
    "spain": ["spain", "madrid", "barcelona", "valencia"],
    "india": ["india", "bangalore", "bengaluru", "delhi", "mumbai", "pune", "hyderabad", "gurgaon", "gurugram", "noida", "chennai"],
    "singapore": ["singapore", "apac"],
}
ROLE_ABBREVIATIONS = {"pm": "product manager", "apm": "associate product manager", "gpm": "group product manager",
                      "tpm": "technical program manager", "cos": "chief of staff", "em": "engineering manager",
                      "genai": "generative ai", "ml": "machine learning"}
ROLE_STOPWORDS = {"of", "and", "the", "a", "an", "&", "-", "to", "senior", "sr", "junior", "jr", "lead", "head", "principal", "staff"}

ATS_PATTERNS = [
    ("Greenhouse", re.compile(r"(?:boards|job-boards)(?:\.eu)?\.greenhouse\.io/(?:embed/job_board(?:/js)?\?for=)?([A-Za-z0-9_-]+)")),
    ("Lever", re.compile(r"jobs\.lever\.co/([A-Za-z0-9_.-]+)")),
    ("Ashby", re.compile(r"jobs\.ashbyhq\.com/([A-Za-z0-9_.%-]+)")),
    ("SmartRecruiters", re.compile(r"(?:jobs|careers)\.smartrecruiters\.com/([A-Za-z0-9_-]+)")),
    ("Workable", re.compile(r"apply\.workable\.com/([A-Za-z0-9_-]+)")),
    ("Workday", re.compile(r"([a-z0-9-]+)\.(wd\d+)\.myworkdayjobs\.com/(?:[a-z]{2}-[A-Z]{2}/)?([A-Za-z0-9_-]+)")),
]
BAD_SLUGS = {"embed", "js", "v1", "api", "jobs", "careers", "en-us", "wday", "search"}


def parse_geo(geo: str):
    """'European Union (Remote)' -> ('European Union', True); 'Remote' -> ('', True)."""
    remote = bool(re.search(r"\bremote\b", geo, re.I))
    region = re.sub(r"(?i)\bremote\b", " ", geo)
    region = re.sub(r"[()\[\]]", " ", region)
    region = re.sub(r"\s+", " ", region).strip(" -,/|")
    return region, remote


def _place_keys(region: str) -> List[str]:
    r = region.lower().strip()
    return LOCATION_ALIASES.get(r, [r] + [w for w in re.findall(r"[a-z]+", r) if len(w) > 3])


def location_matches(job_location: str, geo: str) -> bool:
    loc = (job_location or "").lower()
    if not loc:
        return True  # careers page didn't state a location; let the scorer decide
    region, remote = parse_geo(geo)
    if remote and not re.search(r"remote|anywhere|distributed|work from home|wfh", loc):
        return False
    if not region:
        return True
    if any(re.search(rf"\b{re.escape(k)}\b", loc) for k in _place_keys(region)):
        return True
    # A plain "Remote" posting with no country attached is kept for remote searches
    return remote and bool(re.fullmatch(r"[\s\W]*(fully\s+)?(remote|anywhere)[\s\W]*", loc))


def _normalize_title(text: str) -> str:
    t = (text or "").lower().replace("/", " ")
    for abbr, full in ROLE_ABBREVIATIONS.items():
        t = re.sub(rf"\b{abbr}\b", full, t)
    return t


def expand_role_variants(roles: List[str]) -> List[str]:
    """Adds similar titles for each target role (via local Qwen) so 'AI PM' also catches 'Product Manager, AI'."""
    variants = [_normalize_title(r) for r in roles]
    prompt = f"""For each of these target job titles, list up to 4 closely similar job titles that a company might use
for the same job (same seniority band and function). Target titles: {', '.join(roles)}
Return STRICTLY a JSON array of strings."""
    try:
        match = re.search(r"\[.*\]", ollama_llm.invoke(prompt).content, re.DOTALL)
        variants += [_normalize_title(v) for v in (json.loads(match.group(0)) if match else []) if isinstance(v, str)]
    except Exception as e:
        print(f"  ⚠️ Role variant notice: {e}")
    return list(dict.fromkeys(v.strip() for v in variants if v.strip()))


_ROLE_VARIANT_CACHE: Dict[tuple, List[str]] = {}
_ROLE_VARIANT_LOCK = __import__("threading").Lock()

def expand_role_variants_cached(roles: List[str]) -> List[str]:
    """Parallel branches share one Qwen call for similar titles."""
    with _ROLE_VARIANT_LOCK:
        key = tuple(roles)
        if key not in _ROLE_VARIANT_CACHE:
            _ROLE_VARIANT_CACHE[key] = expand_role_variants(roles)
        return _ROLE_VARIANT_CACHE[key]


def title_matches_roles(title: str, roles: List[str]) -> bool:
    t = _normalize_title(title)
    for role in roles:
        role_l = _normalize_title(role).strip()
        if role_l and role_l in t:
            return True
        tokens = [w for w in re.findall(r"[a-z0-9]+", role_l) if w not in ROLE_STOPWORDS]
        if tokens and all(re.search(rf"\b{re.escape(w)}", t) for w in tokens):
            return True
    return False


def _parse_iso(value) -> datetime:
    if not value:
        return None
    try:
        if isinstance(value, (int, float)):  # Lever uses epoch milliseconds
            return datetime.fromtimestamp(value / 1000)
        return datetime.fromisoformat(str(value).replace("Z", "+00:00")).replace(tzinfo=None)
    except Exception:
        return None


def _parse_workday_posted(text: str) -> datetime:
    txt = (text or "").lower()
    if "today" in txt:
        return datetime.now()
    if "yesterday" in txt:
        return datetime.now() - timedelta(days=1)
    m = re.search(r"(\d+)\+?\s*day", txt)
    return datetime.now() - timedelta(days=int(m.group(1))) if m else None


def _root_url(url: str) -> str:
    p = urllib.parse.urlparse(url if "://" in url else f"https://{url}")
    return f"{p.scheme or 'https'}://{p.netloc}"


def _clean_company_name(title: str, url: str) -> str:
    name = re.split(r"\s+[\-|–—:·]\s+", (title or "").strip())[0].strip()
    if not name or len(name) > 50:
        name = urllib.parse.urlparse(url).netloc.replace("www.", "").split(".")[0].title()
    return name


def _match_ats(text: str):
    for name, pat in ATS_PATTERNS:
        m = pat.search(text or "")
        if m and m.group(1).lower() not in BAD_SLUGS:
            return name, list(m.groups())
    return None, None


def derive_domain_from_resume(persona_key: str, resume_summary: str) -> str:
    """Reads the candidate's current industry domain from the cached CV summary (cached per persona)."""
    cache = load_cache()
    if cache.get(persona_key, {}).get("domain"):
        return cache[persona_key]["domain"]

    prompt = f"""From this candidate profile, name the ONE industry domain they currently work in,
in 2-4 words (e.g. 'Fintech', 'B2B SaaS', 'EdTech', 'Healthcare AI'). Return only the domain, nothing else.

Candidate Profile:
{resume_summary}"""
    try:
        domain = ollama_llm.invoke(prompt).content.strip().splitlines()[0].strip(" .'\"*")[:60]
    except Exception:
        domain = ""
    domain = domain or "Technology"

    if persona_key in cache:
        cache[persona_key]["domain"] = domain
        save_cache(cache)
    return domain


def discover_hiring_companies(roles: List[str], geo: str, domain: str, limit: int, exclude: set) -> List[Dict[str, Any]]:
    """Role-first shortlist: search postings for the role in the area, keep the companies behind them."""
    region, remote = parse_geo(geo)
    where = f"in {region}" if region else ""
    mode = "remote" if remote else ""
    found: Dict[str, Dict[str, Any]] = {}

    for role in roles:
        queries = [f"{mode} {role} {domain} job {where}", f"{mode} {role} job opening {where}"]
        for q in queries:
            q = re.sub(r"\s+", " ", q).strip()
            # a) Postings on public ATS boards -> company + its board, no website lookup needed
            try:
                res = exa.search(query=q, type="auto", include_domains=ATS_DOMAINS, num_results=25)
                for r in res.results:
                    ats, slug = _match_ats(r.url)
                    key = f"{ats}:{'/'.join(slug or [])}".lower()
                    if ats and key not in exclude and key not in found:
                        found[key] = {"name": slug[0].replace("-", " ").title(), "website": "", "ats": ats,
                                      "slug": slug, "key": key, "source": "role posting"}
            except Exception as e:
                print(f"  ⚠️ Exa ATS posting search issue for '{q}': {e}")
            # b) Postings on companies' own websites
            try:
                res = exa.search(query=f"{q} careers", type="auto", exclude_domains=ATS_DOMAINS + JOB_BOARD_DOMAINS, num_results=15)
                for r in res.results:
                    ats, slug = _match_ats(r.url)
                    site = _root_url(r.url)
                    if ats or site in exclude or site in found or any(s in site for s in SKIP_COMPANY_DOMAINS):
                        continue
                    found[site] = {"name": urllib.parse.urlparse(site).netloc.replace("www.", "").split(".")[0].title(),
                                   "website": site, "key": site, "source": "role posting"}
            except Exception as e:
                print(f"  ⚠️ Exa web posting search issue for '{q}': {e}")
            if len(found) >= limit:
                break
    return list(found.values())[:limit]


def shortlist_domain_companies(domain: str, geo: str, roles: List[str], count: int, exclude: set) -> List[Dict[str, Any]]:
    """Domain shortlist: companies in `domain` headquartered in / with offices in the area (Exa, then local Qwen)."""
    region, remote = parse_geo(geo)
    where = f"headquartered in or with offices in {region}" if region else "remote-first"
    hiring = " hiring remote" if remote else " hiring"
    found: Dict[str, Dict[str, Any]] = {}
    queries = [f"{domain} company {where}"] + [f"{domain} company {where}{hiring} {role}" for role in roles[:2]]

    for q in queries:
        if len(found) >= count:
            break
        try:
            res = exa.search(query=q, type="auto", category="company", num_results=count * 2)
            for r in res.results:
                site = _root_url(r.url)
                if site in exclude or site in found or any(s in site for s in SKIP_COMPANY_DOMAINS):
                    continue
                found[site] = {"name": _clean_company_name(r.title, r.url), "website": site, "key": site, "source": "domain"}
                if len(found) >= count:
                    break
        except Exception as e:
            print(f"  ⚠️ Exa company search issue for '{q}': {e}")

    if len(found) < count:
        prompt = f"""List {count - len(found)} real {domain} companies {where} that hire for: {', '.join(roles)}{' (remote)' if remote else ''}.
Return STRICTLY a JSON array: [{{"name": "Company", "website": "https://company.com"}}]"""
        try:
            match = re.search(r"\[.*\]", ollama_llm.invoke(prompt).content, re.DOTALL)
            for c in (json.loads(match.group(0)) if match else []):
                site = c.get("website") or ""
                if not site:
                    res = exa.search(query=f"{c.get('name')} official website", type="auto", category="company", num_results=1)
                    site = res.results[0].url if res.results else ""
                if not site:
                    continue
                site = _root_url(site)
                if site not in exclude and site not in found:
                    found[site] = {"name": c.get("name") or _clean_company_name("", site), "website": site, "key": site, "source": "domain"}
        except Exception as e:
            print(f"  ⚠️ Qwen company shortlist notice: {e}")

    return list(found.values())[:count]


def detect_career_source(website: str):
    """Visits the homepage and common careers paths; returns (ats_name, slug_parts, careers_page_url)."""
    candidates, careers_url = [], None
    try:
        home = requests.get(website, headers=CAREER_HEADERS, timeout=6)
        if home.status_code == 200:
            ats, slug = _match_ats(home.text)
            if ats:
                return ats, slug, home.url
            soup = BeautifulSoup(home.text, "html.parser")
            for a in soup.find_all("a", href=True):
                if re.search(r"career|jobs|join", a["href"] + " " + a.get_text(" ", strip=True), re.I):
                    candidates.append(urllib.parse.urljoin(home.url, a["href"]))
                if len(candidates) >= 2:
                    break
    except Exception:
        pass
    candidates += [website.rstrip("/") + p for p in CAREER_PATHS]

    for url in dict.fromkeys(candidates):
        ats, slug = _match_ats(url)  # careers link may itself point to the ATS
        if ats:
            return ats, slug, url
        try:
            r = requests.get(url, headers=CAREER_HEADERS, timeout=6)
        except Exception:
            continue
        if r.status_code != 200:
            continue
        ats, slug = _match_ats(r.url + " " + r.text)
        if ats:
            return ats, slug, r.url
        careers_url = careers_url or r.url
    return None, None, careers_url


def guess_ats_from_name(company: str, website: str):
    """Many boards use the company name as slug; probe the common public APIs directly."""
    root = urllib.parse.urlparse(website).netloc.replace("www.", "").split(".")[0]
    for slug in dict.fromkeys([re.sub(r"[^a-z0-9]", "", company.lower()), root]):
        if not slug:
            continue
        for ats, url in [("Greenhouse", f"https://boards-api.greenhouse.io/v1/boards/{slug}/jobs"),
                         ("Lever", f"https://api.lever.co/v0/postings/{slug}?mode=json&limit=1"),
                         ("Ashby", f"https://api.ashbyhq.com/posting-api/job-board/{slug}")]:
            try:
                r = requests.get(url, timeout=5)
                if r.status_code == 200 and r.text.strip() not in ("", "[]"):
                    return ats, [slug]
            except Exception:
                pass
    return None, None


def fetch_ats_jobs(ats: str, slug_parts, roles: List[str]) -> List[Dict[str, Any]]:
    """Pulls open postings from a public ATS job-board API."""
    slug = slug_parts[0]
    out = []
    if ats == "Greenhouse":
        r = requests.get(f"https://boards-api.greenhouse.io/v1/boards/{slug}/jobs", timeout=8)
        for j in r.json().get("jobs", []) if r.status_code == 200 else []:
            out.append({"title": j.get("title"), "location": (j.get("location") or {}).get("name", ""),
                        "url": j.get("absolute_url"), "posted": _parse_iso(j.get("first_published") or j.get("updated_at"))})
    elif ats == "Lever":
        r = requests.get(f"https://api.lever.co/v0/postings/{slug}?mode=json", timeout=8)
        for j in r.json() if r.status_code == 200 else []:
            cat = j.get("categories") or {}
            loc = cat.get("location", "") + (" / Remote" if j.get("workplaceType") == "remote" else "")
            out.append({"title": j.get("text"), "location": loc, "url": j.get("hostedUrl"), "posted": _parse_iso(j.get("createdAt"))})
    elif ats == "Ashby":
        r = requests.get(f"https://api.ashbyhq.com/posting-api/job-board/{slug}", timeout=8)
        for j in r.json().get("jobs", []) if r.status_code == 200 else []:
            loc = (j.get("location") or "") + (" / Remote" if j.get("isRemote") else "")
            out.append({"title": j.get("title"), "location": loc, "url": j.get("jobUrl"), "posted": _parse_iso(j.get("publishedAt"))})
    elif ats == "SmartRecruiters":
        r = requests.get(f"https://api.smartrecruiters.com/v1/companies/{slug}/postings?limit=100", timeout=8)
        for j in r.json().get("content", []) if r.status_code == 200 else []:
            l = j.get("location") or {}
            loc = ", ".join(x for x in [l.get("city"), l.get("country")] if x) + (" / Remote" if l.get("remote") else "")
            out.append({"title": j.get("name"), "location": loc, "url": f"https://jobs.smartrecruiters.com/{slug}/{j.get('id')}",
                        "posted": _parse_iso(j.get("releasedDate"))})
    elif ats == "Workable":
        r = requests.get(f"https://apply.workable.com/api/v1/widget/accounts/{slug}", timeout=8)
        for j in r.json().get("jobs", []) if r.status_code == 200 else []:
            loc = ", ".join(x for x in [j.get("city"), j.get("country")] if x) + (" / Remote" if j.get("telecommuting") else "")
            out.append({"title": j.get("title"), "location": loc, "url": j.get("url") or j.get("shortlink"),
                        "posted": _parse_iso(j.get("published_on") or j.get("created_at"))})
    elif ats == "Workday":
        tenant, wd, site = slug_parts
        base = f"https://{tenant}.{wd}.myworkdayjobs.com"
        for role in roles:
            r = requests.post(f"{base}/wday/cxs/{tenant}/{site}/jobs", timeout=8, headers=CAREER_HEADERS,
                              json={"appliedFacets": {}, "limit": 20, "offset": 0, "searchText": role})
            for j in r.json().get("jobPostings", []) if r.status_code == 200 else []:
                out.append({"title": j.get("title"), "location": j.get("locationsText", ""),
                            "url": f"{base}/en-US/{site}{j.get('externalPath', '')}", "posted": _parse_workday_posted(j.get("postedOn"))})
    return out


def scrape_generic_career_page(careers_url: str, roles: List[str]) -> List[Dict[str, Any]]:
    """Plain-HTML careers page: keep links whose text looks like one of the target roles."""
    out = []
    try:
        r = requests.get(careers_url, headers=CAREER_HEADERS, timeout=8)
        soup = BeautifulSoup(r.text, "html.parser")
        for a in soup.find_all("a", href=True):
            text = a.get_text(" ", strip=True)
            if 3 < len(text) < 120 and title_matches_roles(text, roles):
                out.append({"title": text, "location": "", "url": urllib.parse.urljoin(r.url, a["href"]), "posted": None})
    except Exception:
        pass
    return out


def fetch_career_page_jobs(company: Dict[str, Any], roles: List[str], search_roles: List[str], geo: str, days: int) -> List[Dict[str, Any]]:
    name, website = company["name"], company.get("website", "")
    ats, slug_parts, careers_url = company.get("ats"), company.get("slug"), None
    if not ats and website:
        ats, slug_parts, careers_url = detect_career_source(website)
        if not ats:
            ats, slug_parts = guess_ats_from_name(name, website)

    if ats == "Greenhouse":  # boards found via postings only know the slug; fetch the real company name
        try:
            name = requests.get(f"https://boards-api.greenhouse.io/v1/boards/{slug_parts[0]}", timeout=5).json().get("name") or name
        except Exception:
            pass

    postings, source = [], ats or "Website"
    try:
        if ats:
            postings = fetch_ats_jobs(ats, slug_parts, search_roles)
        elif careers_url:
            postings = scrape_generic_career_page(careers_url, roles)
    except Exception as e:
        print(f"  ⚠️ Career page read issue for {name}: {e}")

    # Last resort for JS-rendered careers pages: let Exa find role pages on the company's own domain
    if not postings and website:
        try:
            netloc = urllib.parse.urlparse(website).netloc
            for role in search_roles:
                res = exa.search(query=f"{name} {role} job opening", type="auto", include_domains=[netloc], num_results=3)
                for r in res.results:
                    postings.append({"title": (r.title or "").split("|")[0].strip(), "location": "", "url": r.url, "posted": None})
            source = "Website (Exa)"
        except Exception:
            pass

    jobs = finalize_postings([{**p, "company": name} for p in postings], roles, geo, days, f"Career Page ({source})")
    print(f"  {'✓' if jobs else '·'} {name} [{source}]: {len(jobs)} matching role(s)")
    return jobs


# ==========================================
# VC / PE PORTFOLIO + JOB BOARD / HEADHUNTER SEARCH
# ==========================================
DEFAULT_VC_FUNDS = ["Accel", "Matrix Partners"]  # always included; local funds for each area are added automatically

# Local job boards & headhunters per area ("global" is always searched). Add any site here, e.g. "numi.com".
JOB_BOARDS = {
    "global": ["wellfound.com", "jobgether.com", "michaelpage.com", "hays.com", "robertwalters.com", "iimjobs.com",
               "cutshort.io", "indeed.com", "glassdoor.com", "workatastartup.com", "himalayas.app", "remoteok.com",
               "weworkremotely.com", "welcometothejungle.com"],
    "eu": ["jobgether.com", "welcometothejungle.com", "eu-startups.com", "berlinstartupjobs.com", "stepstone.de",
           "michaelpage.de", "michaelpage.fr", "michaelpage.nl", "michaelpage.es", "michaelpage.it",
           "michaelpage.ie", "michaelpage.be", "michaelpage.pt"],
    "uk": ["michaelpage.co.uk", "welcometothejungle.com", "hays.co.uk", "robertwalters.co.uk", "cwjobs.co.uk"],
    "netherlands": ["michaelpage.nl", "welcometothejungle.com"],
    "spain": ["michaelpage.es", "infojobs.net", "welcometothejungle.com"],
    "india": ["naukri.com", "iimjobs.com", "instahyre.com", "cutshort.io", "hirist.tech", "foundit.in", "michaelpage.co.in"],
    "singapore": ["michaelpage.com.sg", "mycareersfuture.gov.sg", "jobstreet.com.sg", "hays.com.sg"],
}
JOB_BOARDS["european union"] = JOB_BOARDS["eu"]
JOB_BOARDS["europe"] = JOB_BOARDS["eu"] + JOB_BOARDS["uk"]
JOB_BOARDS["london"] = JOB_BOARDS["uk"]
HEADHUNTERS = {"michaelpage": "Michael Page", "hays": "Hays", "robertwalters": "Robert Walters"}
# Hiring posts on social sites ("we're hiring an AI PM ...")
SOCIAL_SOURCES = {"X/Twitter": ["x.com", "twitter.com"], "Reddit": ["reddit.com"]}
SOCIAL_DOMAINS = ["linkedin.", "twitter.", "x.com", "facebook.", "instagram.", "youtube.", "medium.", "github.", "apple.com", "google."]


def boards_for_geo(geo: str) -> List[str]:
    region, _ = parse_geo(geo)
    r = region.lower()
    boards = list(JOB_BOARDS["global"])
    for key, sites in JOB_BOARDS.items():
        if key != "global" and r and (r == key or r in LOCATION_ALIASES.get(key, [])):
            boards += sites
    return list(dict.fromkeys(boards))


def _board_label(url: str) -> str:
    netloc = urllib.parse.urlparse(url).netloc.replace("www.", "")
    if any(d in netloc for d in ATS_DOMAINS):
        return f"ATS ({netloc})"
    for key, name in HEADHUNTERS.items():
        if key in netloc:
            return f"Recruiter ({name})"
    return f"Job Board ({netloc})"


def _split_board_title(title: str):
    """'AI Product Manager at Acme - Berlin | Jobgether' -> ('AI Product Manager', 'Acme')."""
    t = (title or "").strip()
    m = re.match(r"(.+?)\s+(?:at|@|bei|chez)\s+(.+?)(?:\s+[|\-–—]\s+.*)?$", t)
    if m:
        return m.group(1).strip(), m.group(2).strip()
    parts = re.split(r"\s+[|\-–—]\s+", t)
    company = parts[1].strip() if len(parts) > 2 else "Unknown Company"  # 2-part titles are usually 'Role | Site'
    return parts[0].strip(), company


def _exa_postings(query: str, domains: List[str], days: int, label_fn, num: int = 25) -> List[Dict[str, Any]]:
    """Exa search restricted to `domains`; every result is treated as one posting."""
    out = []
    try:
        res = exa.search(query=query, type="auto", include_domains=domains, num_results=num)
        for r in res.results:
            title, company = _split_board_title(r.title)
            ats, slug = _match_ats(r.url)
            if ats:  # ATS posting URLs carry the company in the slug
                company = slug[0].replace("-", " ").title()
            out.append({"title": title, "company": company, "location": "", "url": r.url,
                        "posted": _parse_iso(getattr(r, "published_date", None)), "platform": label_fn(r.url)})
    except Exception as e:
        print(f"  ⚠️ Exa board search issue for '{query}': {e}")
    return out


def _jobs_from_next_data(html: str, base_url: str) -> List[Dict[str, Any]]:
    """Portfolio job boards (Getro / Consider) ship their job list as JSON inside the page."""
    m = re.search(r'<script id="__NEXT_DATA__"[^>]*>(.*?)</script>', html, re.S)
    if not m:
        return []
    out, stack = [], [json.loads(m.group(1))]
    while stack:
        node = stack.pop()
        if isinstance(node, list):
            stack.extend(node)
        elif isinstance(node, dict):
            url = node.get("url") or node.get("applicationUrl") or node.get("applyUrl")
            if isinstance(node.get("title"), str) and isinstance(url, str):
                org = node.get("organization") or node.get("company") or {}
                locs = node.get("locations") or node.get("location") or []
                locs = locs if isinstance(locs, list) else [locs]
                created = node.get("createdAt") or node.get("created_at") or node.get("postedAt")
                if isinstance(created, (int, float)) and created < 1e11:  # epoch seconds -> ms
                    created *= 1000
                out.append({"title": node["title"],
                            "company": (org.get("name") if isinstance(org, dict) else str(org)) or "Unknown Company",
                            "location": ", ".join(l.get("name", "") if isinstance(l, dict) else str(l) for l in locs),
                            "url": urllib.parse.urljoin(base_url, url), "posted": _parse_iso(created)})
            stack.extend(v for v in node.values() if isinstance(v, (dict, list)))
    return out


def find_vc_funds(geo: str, named_funds: List[str]) -> List[Dict[str, str]]:
    """Named funds (Accel, Matrix Partners, user's own) + VC/PE funds headquartered in the area."""
    region, _ = parse_geo(geo)
    funds: Dict[str, Dict[str, str]] = {}
    queries = [(f"{f} venture capital fund {region}".strip(), 1) for f in named_funds]
    if region:
        queries += [(f"venture capital fund headquartered in {region} investing in startups", 5),
                    (f"private equity growth fund headquartered in {region}", 3)]
    for q, n in queries:
        try:
            res = exa.search(query=q, type="auto", category="company", num_results=n)
            for r in res.results:
                site = _root_url(r.url)
                if site not in funds and not any(s in site for s in SKIP_COMPANY_DOMAINS + SOCIAL_DOMAINS):
                    funds[site] = {"name": _clean_company_name(r.title, r.url), "website": site}
        except Exception as e:
            print(f"  ⚠️ Fund search issue for '{q}': {e}")
    return list(funds.values())


def explore_vc_fund(fund: Dict[str, str], max_companies: int) -> Dict[str, Any]:
    """Finds the fund's portfolio job board and, if it has none, its portfolio companies' websites."""
    board, portfolio_url, companies = None, None, []
    fund_netloc = urllib.parse.urlparse(fund["website"]).netloc.replace("www.", "")
    try:
        home = requests.get(fund["website"], headers=CAREER_HEADERS, timeout=6)
        soup = BeautifulSoup(home.text, "html.parser")
        for a in soup.find_all("a", href=True):
            href = urllib.parse.urljoin(home.url, a["href"])
            text = (a.get_text(" ", strip=True) + " " + href).lower()
            if not board and re.search(r"jobs|careers|talent|getro|consider", text) and re.search(r"portfolio|jobs\.|talent", text):
                board = _root_url(href) if "jobs." in href or "getro" in href else href
            if not portfolio_url and re.search(r"portfolio|companies", text) and fund_netloc in href:
                portfolio_url = href
    except Exception:
        pass

    if not board and portfolio_url:
        try:
            page = requests.get(portfolio_url, headers=CAREER_HEADERS, timeout=8)
            for a in BeautifulSoup(page.text, "html.parser").find_all("a", href=True):
                href = urllib.parse.urljoin(page.url, a["href"])
                site = _root_url(href)
                if href.startswith("http") and fund_netloc not in site and not any(s in site for s in SOCIAL_DOMAINS + SKIP_COMPANY_DOMAINS):
                    if site not in [c["website"] for c in companies]:
                        companies.append({"name": a.get_text(" ", strip=True)[:50] or _clean_company_name("", site),
                                          "website": site, "key": site, "source": f"VC portfolio: {fund['name']}"})
                if len(companies) >= max_companies:
                    break
        except Exception:
            pass
    return {"fund": fund["name"], "board": board, "companies": companies}


def fetch_fund_board_jobs(board: str, fund_name: str, roles: List[str], search_roles: List[str], geo: str, days: int) -> List[Dict[str, Any]]:
    """Reads a VC portfolio job board (Getro / Consider style), falling back to Exa on the board's domain."""
    postings = []
    for role in search_roles:
        try:
            r = requests.get(f"{board.rstrip('/')}/jobs?q={urllib.parse.quote(role)}", headers=CAREER_HEADERS, timeout=8)
            postings += _jobs_from_next_data(r.text, r.url) if r.status_code == 200 else []
        except Exception:
            pass
    if not postings:
        netloc = urllib.parse.urlparse(board).netloc
        region, remote = parse_geo(geo)
        for role in search_roles:
            postings += _exa_postings(f"{'remote ' if remote else ''}{role} job {region}", [netloc], days, lambda u: "")
    jobs = finalize_postings(postings, roles, geo, days, f"VC Portfolio Board ({fund_name})")
    print(f"  {'✓' if jobs else '·'} {fund_name} portfolio board: {len(jobs)} matching role(s)")
    return jobs


def finalize_postings(postings: List[Dict[str, Any]], roles: List[str], geo: str, days: int, platform: str) -> List[Dict[str, Any]]:
    """Shared role / location / recency filter that maps postings into the pipeline's job format."""
    cutoff = datetime.now() - timedelta(days=days)
    jobs = []
    for p in postings:
        if not p.get("url") or not title_matches_roles(p.get("title"), roles):
            continue
        if not location_matches(p.get("location"), geo):
            continue
        if p.get("posted") and p["posted"] < cutoff:
            continue
        jobs.append({
            "Job Profile": p["title"],
            "Target Company": p.get("company") or "Unknown Company",
            "Location": p.get("location") or geo,
            "Job Link": p["url"],
            "Platform Name": p.get("platform") or platform,
            "Date Posted": p["posted"].strftime("%Y-%m-%d") if p.get("posted") else "Not listed"
        })
    return jobs


# Node 2c: ATS boards, job boards, headhunters & social hiring posts (rest of the web agent)
def search_job_boards_node(state: PipelineState) -> Dict[str, Any]:
    if state.get("search_mode", "both") == "linkedin":
        return {}
    days = state.get("job_duration_days", 3)
    search_roles = [_normalize_title(r) for r in state["roles"]]
    variants = expand_role_variants_cached(state["roles"])
    print(f"\n📋 Node 2c: Searching ATS boards, job boards, headhunters, Reddit & X (past {days} days)...")

    jobs = []
    for geo in state["target_geos"]:
        region, remote = parse_geo(geo)
        boards = boards_for_geo(geo)
        found, social = [], []
        for role in search_roles:
            q = f"{'remote ' if remote else ''}{role} job {region}".strip()
            found += _exa_postings(q, ATS_DOMAINS, days, _board_label)
            found += _exa_postings(q, boards, days, _board_label)
            for name, domains in SOCIAL_SOURCES.items():
                social += _exa_postings(f"hiring {q}", domains, days, lambda u, n=name: f"Social ({n})", num=10)
        geo_jobs = finalize_postings(found, variants, geo, days, "Job Board")
        # Social posts have no posting date in the title, so keep only ones Exa dates inside the window
        geo_jobs += finalize_postings([p for p in social if p.get("posted")], variants, geo, days, "Social")
        print(f"  ✓ {geo}: {len(geo_jobs)} posting(s) from ATS boards, {len(boards)} job sites/recruiters, Reddit & X")
        jobs += geo_jobs
    return {"board_jobs": jobs}

# Fan-out point: LinkedIn branch and career-page branch start from here in parallel
def start_search_node(state: PipelineState) -> Dict[str, Any]:
    mode = state.get("search_mode", "both")
    print(f"\n🔀 Starting search branches: {'LinkedIn agent + web agent (parallel)' if mode == 'both' else mode + ' agent only'}")
    return {}


# Node 2a: Company Shortlist (role postings in the area + domain companies)
def shortlist_companies_node(state: PipelineState) -> Dict[str, Any]:
    if state.get("search_mode", "both") == "linkedin":
        return {}

    domain = state.get("target_domain") or derive_domain_from_resume(state.get("candidate_name", "Default"), state.get("resume_summary", ""))
    variants = expand_role_variants_cached(state["roles"])
    count = state.get("career_company_count", 15)
    already = state.get("shortlisted_companies", [])
    seen = {c["key"] for c in already}
    print(f"\n🏢 Node 2a: Shortlisting companies for {state['roles']} in '{domain}' ({count} per location)...")
    print(f"  ↳ Matching similar titles: {', '.join(variants[:8])}")

    named_funds = state.get("vc_funds") or DEFAULT_VC_FUNDS
    known_boards = {(b["board"], b["geo"]) for b in state.get("fund_boards", [])}
    fund_boards, batch = [], []
    for geo in state["target_geos"]:
        # VC / PE funds: portfolio job boards, or portfolio companies when a fund has no board
        funds = find_vc_funds(geo, named_funds)
        with ThreadPoolExecutor(max_workers=6) as pool:
            explored = list(pool.map(lambda f: explore_vc_fund(f, max(3, count // 3)), funds))
        for e in explored:
            if e["board"] and (e["board"], geo) not in known_boards:
                fund_boards.append({"board": e["board"], "fund": e["fund"], "geo": geo})
                known_boards.add((e["board"], geo))
        portfolio = [c for e in explored for c in e["companies"] if c["key"] not in seen][:count]
        seen.update(c["key"] for c in portfolio)
        for c in portfolio:
            c["geo"] = geo
        batch.extend(portfolio)
        fund_notes = [f"{e['fund']} (" + ("job board" if e["board"] else f"{len(e['companies'])} portfolio cos") + ")" for e in explored]
        print(f"  ✓ {geo}: VC/PE funds → " + (", ".join(fund_notes) or "none found"))

        # Role-first gets most of the slots: these companies are proven to be hiring for the role in the area
        hiring = discover_hiring_companies(state["roles"], geo, domain, max(1, count * 2 // 3), seen)
        seen.update(c["key"] for c in hiring)
        domain_cos = shortlist_domain_companies(domain, geo, state["roles"], count - len(hiring), seen)
        seen.update(c["key"] for c in domain_cos)
        for c in hiring + domain_cos:
            c["geo"] = geo
        batch.extend(hiring + domain_cos)
        print(f"  ✓ {geo}: {len(hiring)} hiring for the role, {len(domain_cos)} from domain → "
              + ", ".join(c["name"] for c in hiring + domain_cos))

    return {"target_domain": domain, "role_variants": variants, "shortlisted_companies": already + batch,
            "company_batch": batch, "fund_boards": state.get("fund_boards", []) + fund_boards, "new_fund_boards": fund_boards}


# Node 2b: Company Career Page Search
def search_career_pages_node(state: PipelineState) -> Dict[str, Any]:
    if state.get("search_mode", "both") == "linkedin":
        return {}

    batch = state.get("company_batch", [])
    days = state.get("job_duration_days", 3)
    variants = state.get("role_variants") or state["roles"]
    search_roles = [_normalize_title(r) for r in state["roles"]]
    print(f"\n🌐 Node 2b: Reading career pages of {len(batch)} companies (past {days} days)...")

    with ThreadPoolExecutor(max_workers=8) as pool:
        results = list(pool.map(lambda c: fetch_career_page_jobs(c, variants, search_roles, c["geo"], days), batch))

        board_results = list(pool.map(lambda b: fetch_fund_board_jobs(b["board"], b["fund"], variants, search_roles, b["geo"], days),
                                      state.get("new_fund_boards", [])))

    jobs = [j for res in results + board_results for j in res]
    print(f"  ✓ {len(jobs)} career-page + VC portfolio postings found.")
    return {"career_jobs": jobs}


# Join point: merge both branches before scoring
def merge_search_results_node(state: PipelineState) -> Dict[str, Any]:
    all_jobs = state.get("raw_jobs", []) + state.get("career_jobs", []) + state.get("board_jobs", [])
    unique_map = {j["Job Link"]: j for j in all_jobs if j.get("Job Link")}
    print(f"\n🧩 Merged {len(unique_map)} unique postings (LinkedIn + web agent) for scoring.")
    return {"raw_jobs": list(unique_map.values()), "career_jobs": [], "board_jobs": []}


# Direct Headless Claude Pro Evaluation (Windows-safe list args)
# Direct Headless Claude Pro Evaluation (streams scores live; keeps partial results)
def call_claude_pro_evaluation(resume_summary: str, jobs_payload: str, timeout_s: int = 270) -> Dict[int, Any]:
    import threading
    if not os.path.exists(CLAUDE_EXE):
        raise RuntimeError("Claude CLI executable not found at specified path.")

    total = len(re.findall(r"^ID \d+:", jobs_payload, re.M))
    prompt = f"""[CRITICAL INSTRUCTION: You are an automated batch evaluation script. Do NOT greet. Do NOT offer services. Output ONLY JSON Lines: exactly one JSON object per line, one line per job, in ID order. No array brackets, no markdown, nothing else.]

Candidate Profile:
{resume_summary}

Jobs to Evaluate:
{jobs_payload}

Scoring Rules:
1. Product Manager, AI Product Builder, Strategy, or Chief of Staff roles that align with the candidate MUST score between 75 and 95.
2. Only flag pure software engineering (hands-on coding), ML model research/training, or low-level IT support as 'Low' (score < 50).
3. If a job fits product ownership, leadership, or business operations, assign 'Good' or 'Perfect'.

Output format (one line per job):
{{"id": 1, "match_score": 92, "rating": "Perfect", "rationale": "Direct fit for AI PM role.", "low_match_reason": null}}
{{"id": 2, "match_score": 40, "rating": "Low", "rationale": "Requires ML research.", "low_match_reason": "Role demands ML systems engineering rather than PM leadership."}}
"""
    results = {}

    def record(line: str):
        try:
            item = json.loads(line.strip().strip(","))
            if isinstance(item, dict) and "id" in item and int(item["id"]) not in results:
                results[int(item["id"])] = item
                print(f"\r  🧮 Claude scored {len(results)}/{total} jobs", end="", flush=True)
        except Exception:
            pass

    # List argument structure prevents cmd.exe from truncating multi-line strings
    proc = subprocess.Popen(
        [CLAUDE_EXE, "-p", prompt, "--tools", "", "--output-format", "stream-json", "--verbose", "--include-partial-messages"],
        stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True, encoding="utf-8"
    )
    timer = threading.Timer(timeout_s, proc.kill)  # hard stop after timeout_s seconds
    timer.start()
    buffer = ""
    try:
        for raw in proc.stdout:
            try:
                event = json.loads(raw)
            except Exception:
                continue
            if event.get("type") == "stream_event":  # live text as Claude writes it
                delta = event.get("event", {}).get("delta", {})
                if delta.get("type") == "text_delta":
                    buffer += delta.get("text", "")
                    *complete, buffer = buffer.split("\n")
                    for line in complete:
                        record(line)
            elif event.get("type") == "result":  # final full answer (also covers CLIs without live text)
                for line in str(event.get("result", "")).split("\n"):
                    record(line)
        record(buffer)
    finally:
        timer.cancel()
        proc.wait()
    print()

    if not results:
        raise RuntimeError(f"Claude CLI returned no scores (error, or stopped after {timeout_s}s).")

    # Claude stopped partway: keep its scores, let Qwen score only the rest
    missing = [l for l in jobs_payload.split("\n")
               if re.match(r"ID (\d+):", l) and int(re.match(r"ID (\d+):", l).group(1)) not in results]
    if missing:
        print(f"  ⚠️ Claude stopped at {len(results)}/{total}; Qwen scores only the remaining {len(missing)}.")
        results.update(evaluate_with_local_qwen(resume_summary, "\n".join(missing)))
    return results


    
# Working Local Qwen Evaluator Fallback
def evaluate_with_local_qwen(resume_summary: str, jobs_payload: str) -> Dict[int, Any]:
    print("  ⚙️ Invoking Local Qwen 2.5 (7B) to dynamically score all jobs...")
    prompt = f"""Compare this candidate resume profile against each job posting.

Candidate Profile:
{resume_summary}

Jobs to Evaluate:
{jobs_payload}

For each job, evaluate alignment with Product Management / Strategy:
- If the role is Product Manager, Product Lead, or Chief of Staff, assign match_score between 75 and 95.
- If it is pure software engineering or data science research, assign match_score < 50.

Return STRICTLY a JSON array:
[
  {{"id": 1, "match_score": 88, "rating": "Good", "rationale": "Strong fit for Product leadership.", "low_match_reason": null}}
]"""
    resp = ollama_llm.invoke(prompt)
    try:
        match = re.search(r"\[.*\]", resp.content, re.DOTALL)
        eval_list = json.loads(match.group(0)) if match else []
        return {item.get("id"): item for item in eval_list}
    except Exception as e:
        print(f"  ⚠️ Qwen parse notice: {e}")
        return {}


# Node 3: Deep Match Scoring
def evaluate_matches_node(state: PipelineState) -> Dict[str, Any]:
    print("\n⚖️ Node 3: Deep Match Scoring via Claude Pro...")
    raw_jobs = state.get("raw_jobs", [])
    if not raw_jobs:
        return {"evaluated_jobs": [], "low_match_summary": "No jobs found to score."}

    eval_map = {}
    BATCH_SIZE = 20

    for start in range(0, len(raw_jobs), BATCH_SIZE):
        batch = raw_jobs[start:start + BATCH_SIZE]
        jobs_payload = "\n".join([
            f"ID {i+1}: {j.get('Job Profile')} at {j.get('Target Company')} ({j.get('Location')})"
            for i, j in enumerate(batch)
        ])
        print(f"  📦 Scoring jobs {start + 1}-{start + len(batch)} of {len(raw_jobs)}...")

        # 1. Primary: Direct Claude Pro Execution
        try:
            batch_map = call_claude_pro_evaluation(state['resume_summary'], jobs_payload)
            print("  ✓ Batch scored via Claude Pro (Direct CLI)!")
        except Exception as e:
            print(f"  ⚠️ Claude Pro CLI notice: {e}")
            # 2. Working Fallback: Local Qwen 2.5 (this batch only)
            batch_map = evaluate_with_local_qwen(state['resume_summary'], jobs_payload)

        # Batch IDs run 1-20; shift them to positions in the full job list
        for local_id, ev in batch_map.items():
            try:
                eval_map[start + int(local_id)] = ev
            except (TypeError, ValueError):
                pass
    evaluated = []
    low_matches = []

    for i, job in enumerate(raw_jobs):
        ev = eval_map.get(i + 1, {})
        score = ev.get("match_score")
        
        # Safety heuristic if an ID was missed
        if score is None:
            title_lower = job.get("Job Profile", "").lower()
            score = 85 if any(k in title_lower for k in ["product", "chief of staff", "strategy", "builder", "lead"]) else 50

        job["Match Score"] = score
        job["Rating"] = ev.get("rating", "Good" if score >= 70 else "Average")
        job["Rationale"] = ev.get("rationale", "Aligned with core product leadership capabilities.")
        job["low_match_reason"] = ev.get("low_match_reason")

        if job["Match Score"] >= 60:
            evaluated.append(job)
        else:
            low_matches.append(job)

    evaluated.sort(key=lambda x: x.get("Match Score", 0), reverse=True)

    low_match_items = []
    for lm in low_matches[:4]:
        reason = lm.get("low_match_reason") or lm.get("Rationale")
        low_match_items.append(f"• {lm.get('Job Profile')} @ {lm.get('Target Company')} ({lm.get('Match Score')}%): {reason[:120]}")
    low_match_summary = "\n".join(low_match_items) if low_match_items else "No low matches."

    print(f"  ✓ {len(evaluated)} qualified positions approved for HITL sign-off.")
    return {"evaluated_jobs": evaluated, "low_match_summary": low_match_summary}


# Node 4: Batch Human-in-the-Loop (HITL) Sign-Off
def human_review_node(state: PipelineState) -> Dict[str, Any]:
    evaluated = state.get("evaluated_jobs", [])
    already_approved = state.get("approved_jobs", [])
    target = state.get("target_approved_count", 5)

    if not evaluated:
        print("\n⚠️ No new candidate jobs to review in this cycle.")
        return {"approved_jobs": already_approved, "user_decision": "proceed"}

    print("\n" + "="*68)
    print(f"🧑‍💼 HUMAN-IN-THE-LOOP SIGN-OFF (Approved so far: {len(already_approved)}/{target})")
    print("="*68)

    batch = evaluated[:5]
    remaining = evaluated[5:]

    for idx, job in enumerate(batch, 1):
        print(f"\n[{idx}] {job.get('Job Profile')} @ {job.get('Target Company')}")
        print(f"    Location: {job.get('Location')} | Match Score: {job.get('Match Score')}% ({job.get('Rating')})")
        print(f"    Rationale: {job.get('Rationale')}")
        print(f"    Link: {job.get('Job Link')}")

    print("\n" + "-"*68)
    user_input = input(
        f"Enter IDs to APPROVE (e.g., '1, 3'), 'all', 'none', or 'done' to stop: "
    ).strip().lower()

    newly_approved = []
    if user_input == "all":
        newly_approved.extend(batch)
    elif user_input not in ["none", "done", ""]:
        selected_indices = [int(i.strip()) for i in re.findall(r"\d+", user_input)]
        for i in selected_indices:
            if 1 <= i <= len(batch):
                newly_approved.append(batch[i - 1])

    total_approved = already_approved + newly_approved
    print(f"✓ Total Approved: {len(total_approved)}/{target}")

    if user_input == "done" or len(total_approved) >= target:
        decision = "proceed"
    elif remaining:
        decision = "review_more"
    else:
        fetch_more = input("Reviewed all jobs in this batch. Fetch broader live search? (y/n): ").strip().lower()
        decision = "search_more" if fetch_more == "y" else "proceed"

    return {
        "approved_jobs": total_approved,
        "evaluated_jobs": remaining,
        "user_decision": decision
    }


# Node 5: Real Contact Discovery (Exa SERP)





# ==========================================
# 2. UPDATED NODE 5 WITH GETEMAIL.IO ENRICHMENT
# ==========================================
import time

def verify_email_via_getemail(fullname: str, company: str, max_wait_seconds: int = 8) -> str:
    """Verifies live corporate email via GetEmail.io with automatic polling."""
    api_key = os.environ.get("GETEMAIL_API_KEY", "").strip()
    if not api_key or api_key == "your_actual_getemail_api_key_here":
        return None

    clean_domain = re.sub(r'[^a-zA-Z0-9]', '', company).lower()
    find_url = "https://api.getemail.io/v2/find-email"
    params = {
        "api_key": api_key,
        "fullname": fullname,
        "domain": clean_domain,
        "isEnrichToDo": "true"
    }

    try:
        resp = requests.get(find_url, params=params, timeout=12)
        if resp.status_code != 200:
            return None
            
        data = resp.json()
        if data.get("email"):
            return data.get("email")

        # Async polling for live SMTP handshake completion
        search_id = data.get("id")
        if data.get("code") == "EMAIL_FINDING_PROCESSING" and search_id:
            status_url = "https://api.getemail.io/v2/status-find-email"
            status_params = {"api_key": api_key, "id": search_id}
            
            start_time = time.time()
            while time.time() - start_time < max_wait_seconds:
                time.sleep(2.5)
                status_resp = requests.get(status_url, params=status_params, timeout=10)
                if status_resp.status_code == 200:
                    status_data = status_resp.json()
                    if status_data.get("email"):
                        return status_data.get("email")
                    if status_data.get("code") != "EMAIL_FINDING_PROCESSING":
                        break
    except Exception:
        pass

    return None


def find_contacts_node(state: PipelineState) -> Dict[str, Any]:
    print("\n🤝 Node 5: Enriching Approved Jobs with Real Recruiter Contacts & Verified Emails...")
    approved = state.get("approved_jobs", [])

    for job in approved:
        company = job.get("Target Company", "").strip()
        if not company or company in ["N/A", "Unknown Company", "Direct ATS"]:
            job["Prospect_Employer_Name"] = "Recruiting Team"
            job["Prospect_Employer_Title"] = "Talent Acquisition"
            job["Prospect_Employer_Profile_LI"] = "Not Found"
            job["Prospect_Employer_Profile_mail"] = "N/A"
            continue

        clean_comp = re.sub(r'(?i)\b(inc|ltd|pvt|technologies|corporation|llc|corp|co)\b', '', company).strip()
        clean_comp = re.sub(r'[^a-zA-Z0-9\s]', '', clean_comp).strip()
        domain_slug = re.sub(r'[^a-zA-Z0-9]', '', company).lower()

        # Step 1: Find real recruiter profile via Exa
        query = f'site:linkedin.com/in "{clean_comp or company}" ("Talent Acquisition" OR "Technical Recruiter" OR "Head of Product")'
        
        name = "Recruiting Team"
        headline = "Talent Acquisition"
        li_url = "Not Found"

        try:
            res = exa.search(
                query=query,
                type="auto",
                include_domains=["linkedin.com"],
                num_results=2
            )
            for r in res.results:
                if "/in/" in r.url:
                    li_url = r.url
                    raw_title = r.title.replace(" | LinkedIn", "").replace(" - LinkedIn", "")
                    parts = raw_title.split(" - ") if " - " in raw_title else raw_title.split(" | ")
                    name = parts[0].strip()
                    headline = parts[1].strip() if len(parts) > 1 else "Talent Acquisition"
                    break
        except Exception as e:
            print(f"  ⚠️ Recruiter search issue for {company}: {e}")

        # Step 2: Live SMTP Email Verification via GetEmail.io
        verified_email = None
        if name != "Recruiting Team":
            print(f"  📧 Verifying inbox for '{name}' at '{company}'...")
            verified_email = verify_email_via_getemail(fullname=name, company=clean_comp or company)

        # Step 3: Fallback formula if GetEmail has no match or credits are exhausted
        if not verified_email:
            first_clean = re.sub(r'[^a-zA-Z]', '', name.split()[0]).lower()
            if first_clean and len(first_clean) >= 3 and name != "Recruiting Team":
                verified_email = f"{first_clean}@{domain_slug}.com"
            else:
                verified_email = f"hiring@{domain_slug}.com"

        job["Prospect_Employer_Name"] = name
        job["Prospect_Employer_Title"] = headline
        job["Prospect_Employer_Profile_LI"] = li_url
        job["Prospect_Employer_Profile_mail"] = verified_email
        print(f"  ✓ {company}: {name} ({verified_email})")

    return {"approved_jobs": approved}

# Node 6: Deduplicating Excel Exporter
def export_excel_node(state: PipelineState) -> Dict[str, Any]:
    print("\n📊 Node 6: Appending Approved Jobs to Excel Database...")
    approved = state.get("approved_jobs", [])
    if not approved:
        print("  ⚠️ No approved jobs to export.")
        return {}

    new_rows = []
    for j in approved:
        new_rows.append({
            "Resume Applied With": state.get("candidate_name", "Default_CV"),
            "Job Profile": j.get("Job Profile", "N/A"),
            "Target Company": j.get("Target Company", "N/A"),
            "Location": j.get("Location", "Remote"),
            "Match Score": f"{j.get('Match Score', 'N/A')}%",
            "Rating": j.get("Rating", "N/A"),
            "Rationale": j.get("Rationale", "N/A"),
            "Job Link": j.get("Job Link", "N/A"),
            "Date Posted": j.get("Date Posted", state.get("duration_label", "Recent")),
            "Platform Name": j.get("Platform Name", "LinkedIn"),
            "Prospect_Employer_Name": j.get("Prospect_Employer_Name", "N/A"),
            "Prospect_Employer_Title": j.get("Prospect_Employer_Title", "N/A"),
            "Prospect_Employer_Profile_LI": j.get("Prospect_Employer_Profile_LI", "N/A"),
            "Prospect_Employer_Profile_mail": j.get("Prospect_Employer_Profile_mail", "N/A")
        })

    new_df = pd.DataFrame(new_rows)

    if os.path.exists(EXCEL_PATH):
        try:
            existing_df = pd.read_excel(EXCEL_PATH, sheet_name="Matched Jobs")
            final_df = pd.concat([existing_df, new_df], ignore_index=True).drop_duplicates(
                subset=["Job Link"], keep="first"
            )
        except Exception:
            final_df = new_df
    else:
        final_df = new_df

    final_df.to_excel(EXCEL_PATH, index=False, sheet_name="Matched Jobs")
    print(f"  ✓ Successfully saved {len(final_df)} total entries in {EXCEL_PATH}")
    return {}

# ==========================================
# 6. GRAPH COMPILATION
# ==========================================
def hitl_routing(state: PipelineState) -> str:
    decision = state.get("user_decision", "proceed")
    if decision == "review_more":
        return "human_review"
    elif decision == "search_more":
        return "search_jobs"
    return "find_contacts"

workflow = StateGraph(PipelineState)

workflow.add_node("parse_resume", parse_resume_node)
workflow.add_node("start_search", start_search_node)
workflow.add_node("search_jobs", search_jobs_node)
workflow.add_node("shortlist_companies", shortlist_companies_node)
workflow.add_node("search_career_pages", search_career_pages_node)
workflow.add_node("search_job_boards", search_job_boards_node)
workflow.add_node("merge_results", merge_search_results_node)
workflow.add_node("evaluate_matches", evaluate_matches_node)
workflow.add_node("human_review", human_review_node)
workflow.add_node("find_contacts", find_contacts_node)
workflow.add_node("export_excel", export_excel_node)

workflow.add_edge(START, "parse_resume")
workflow.add_edge("parse_resume", "start_search")
# Two search agents run in parallel (LinkedIn | everything outside LinkedIn), then join before scoring
workflow.add_edge("start_search", "search_jobs")
workflow.add_edge("start_search", "shortlist_companies")
workflow.add_edge("shortlist_companies", "search_career_pages")
workflow.add_edge("search_career_pages", "search_job_boards")
workflow.add_edge(["search_jobs", "search_job_boards"], "merge_results")
workflow.add_edge("merge_results", "evaluate_matches")
workflow.add_edge("evaluate_matches", "human_review")

workflow.add_conditional_edges(
    "human_review",
    hitl_routing,
    {
        "human_review": "human_review",
        "search_jobs": "start_search",  # broader search re-runs both branches (next company batch too)
        "find_contacts": "find_contacts"
    }
)

workflow.add_edge("find_contacts", "export_excel")
workflow.add_edge("export_excel", END)

app = workflow.compile()

# ==========================================
# 7. EXECUTION
# ==========================================
if __name__ == "__main__":
    print("\n" + "="*55)
    print("🤖 HITL JOB AGENT (Claude Pro + Exa Contacts + Local CV)")
    print("="*55 + "\n")

    available_cvs = []
    if os.path.exists(RESUMES_DIR):
        available_cvs = [os.path.splitext(f)[0] for f in os.listdir(RESUMES_DIR) if f.endswith(".pdf")]

    if available_cvs:
        print("Available Resumes in Directory:")
        for idx, cv in enumerate(available_cvs, 1):
            print(f"  [{idx}] {cv}")
        choice = input(f"\nSelect Persona (1-{len(available_cvs)}): ").strip()
        selected_persona = available_cvs[int(choice) - 1] if choice.isdigit() and 1 <= int(choice) <= len(available_cvs) else available_cvs[0]
    else:
        selected_persona = "Simar_Dhingra_CV"

    print(f"\n👉 Active Candidate Profile: {selected_persona}\n")

    dur_input = input("Enter posting recency (e.g. '24 hours', '3 days', '1 week') [Default: 1 week]: ").strip() or "1 week"
    loc_input = input("Enter locations (e.g. 'Delhi / Bangalore / European Union (Remote)') [Default: Remote]: ").strip() or "Remote"
    roles_input = input("Enter target roles (e.g. 'Product Manager / Chief of Staff'): ").strip() or "Product Manager"
    target_count = input("How many approved jobs would you like to target? [Default: 5]: ").strip() or "5"

    mode_input = input("Search agents: 'linkedin' only, 'web' only (career pages, VC portfolios, ATS, job boards, recruiters, Reddit, X) or Enter for BOTH in parallel [Default: both]: ").strip().lower()
    search_mode = "linkedin" if mode_input.startswith("l") else "career" if mode_input.startswith(("c", "w", "o")) else "both"
    domain_input, company_count, funds_input = "", "15", ""
    if search_mode != "linkedin":
        domain_input = input("Target domain/industry (e.g. 'Fintech', 'B2B SaaS') [Default: your current domain from CV]: ").strip()
        company_count = input("Companies to shortlist per location [Default: 15]: ").strip() or "15"
        funds_input = input(f"VC/PE funds to include (e.g. 'Accel / Matrix Partners / Sequoia') [Default: {', '.join(DEFAULT_VC_FUNDS)} + local funds]: ").strip()

    inputs = {
        "candidate_name": selected_persona,
        "roles": [r.strip() for r in re.split(r"[/,]", roles_input) if r.strip()],
        "target_geos": [l.strip() for l in re.split(r"[/,]", loc_input) if l.strip()],
        "job_duration_days": parse_duration_to_days(dur_input),
        "duration_label": dur_input,
        "target_approved_count": int(target_count),
        "approved_jobs": [],
        "search_iteration": 1,
        "search_mode": search_mode,
        "target_domain": domain_input,
        "career_company_count": int(company_count) if company_count.isdigit() else 15,
        "shortlisted_companies": [],
        "company_batch": [],
        "career_jobs": [],
        "board_jobs": [],
        "vc_funds": [f.strip() for f in re.split(r"[/,]", funds_input) if f.strip()],
        "fund_boards": [],
        "new_fund_boards": []
    }

    print("\n🚀 Starting Pipeline Execution...\n")
    final_output = app.invoke(inputs)

    print("\n" + "="*55)
    print("📊 LOW MATCH JOBS ANALYSIS (<50 words per role)")
    print("="*55)
    print(final_output.get("low_match_summary"))
    print("\n✓ Process completed! Check Matched_Jobs.xlsx for your approved jobs and verified recruiter contacts.")

C:\Users\simar\AppData\Local\Temp\ipykernel_13416\51269526.py:16: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
C:\Users\simar\AppData\Local\Temp\ipykernel_13416\51269526.py:37: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  ollama_embeddings = OllamaEmbeddings(model="nomic-embed-text", base_url="http://localhost:11434")
C:\Users\simar\AppData\Local\Temp\ipykernel_13416\51269526.py:91: LangChainDeprecationWarning: The class `Chroma` was deprecated in

📦 Indexing updated resume persona: 1792075619258-Simar_CV_Feedback_AI_Europe.pdf into ChromaDB...
  ✓ 1792075619258-Simar_CV_Feedback_AI_Europe.pdf indexed with persona metadata.
📦 Indexing updated resume persona: CV-Simar-Dhingra.pdf into ChromaDB...
  ✓ CV-Simar-Dhingra.pdf indexed with persona metadata.
📦 Indexing updated resume persona: LI_Profile.pdf into ChromaDB...
  ✓ LI_Profile.pdf indexed with persona metadata.

🤖 HITL JOB AGENT (Claude Pro + Exa Contacts + Local CV)

Available Resumes in Directory:
  [1] 1792075619258-Simar_CV_Feedback_AI_Europe
  [2] CV-Simar-Dhingra
  [3] LI_Profile
  [4] Simar_Dhingra_CV
  [5] Simar_Dhingra_CV_PM



Select Persona (1-5):  5



👉 Active Candidate Profile: Simar_Dhingra_CV_PM



Enter posting recency (e.g. '24 hours', '3 days', '1 week') [Default: 1 week]:  24 hours
Enter locations (e.g. 'Delhi / Bangalore / European Union (Remote)') [Default: Remote]:  Spain remote
Enter target roles (e.g. 'Product Manager / Chief of Staff'):  Product Manager
How many approved jobs would you like to target? [Default: 5]:  10
Search agents: 'linkedin' only, 'web' only (career pages, VC portfolios, ATS, job boards, recruiters, Reddit, X) or Enter for BOTH in parallel [Default: both]:  
Target domain/industry (e.g. 'Fintech', 'B2B SaaS') [Default: your current domain from CV]:  
Companies to shortlist per location [Default: 15]:  25
VC/PE funds to include (e.g. 'Accel / Matrix Partners / Sequoia') [Default: Accel, Matrix Partners + local funds]:  


...

✓ Total Approved: 5/10

🧑‍💼 HUMAN-IN-THE-LOOP SIGN-OFF (Approved so far: 5/10)

[1] Trivelta hiring Senior Product Manager (Remote) Job in Remote @ Unknown Company
    Location: Spain remote | Match Score: 90% (Perfect)
    Rationale: Senior PM at Trivelta, a fintech/iGaming platform, strongly matches the candidate's senior PM background in financial services, wallets, partner APIs, and launching new verticals.
    Link: https://www.glassdoor.com/job-listing/senior-product-manager-remote-trivelta-JV_KO0,29_KE30,38.htm?jl=1010133387367

[2] AI Product Engineer @ LIBEEN Smart Housing
    Location: Madrid, Community of Madrid, Spain | Match Score: 88% (Perfect)
    Rationale: Brings together AI product building (LangGraph/CrewAI, n8n, RAG, Cursor) and real estate experience from managing a residential project. The engineering share of the role should be checked.
    Link: https://es.linkedin.com/jobs/view/ai-product-engineer-at-libeen-smart-housing-4482870900

[3] Head of Product @ Ailin.

Enter IDs to APPROVE (e.g., '1, 3'), 'all', 'none', or 'done' to stop:  'all'


✓ Total Approved: 5/10

🧑‍💼 HUMAN-IN-THE-LOOP SIGN-OFF (Approved so far: 5/10)

[1] Product Manager @ Claire Joster
    Location: Greater Barcelona Metropolitan Area | Match Score: 84% (Good)
    Rationale: Product Manager role that fits the candidate's product discovery, A/B testing and roadmap experience. Few details are available about the hiring company.
    Link: https://es.linkedin.com/jobs/view/product-manager-at-claire-joster-4470511443

[2] Senior Product Manager - Barcelona @ Sercanto
    Location: Spain remote | Match Score: 84% (Good)
    Rationale: Senior PM role fits the candidate's track record of leading new verticals and roadmaps.
    Link: https://www.sercanto.es/detail/a/senior-product-manager_barcelona_322030000

[3] Job Application for Senior Product Manager @ Wallapop
    Location: Spain remote | Match Score: 83% (Good)
    Rationale: Senior PM at consumer marketplace; strong match on user engagement, A/B testing, feature launches and wallet/payments experience.
 

Enter IDs to APPROVE (e.g., '1, 3'), 'all', 'none', or 'done' to stop:  all


✓ Total Approved: 10/10

🤝 Node 5: Enriching Approved Jobs with Real Recruiter Contacts & Verified Emails...
  ✓ Bjakcareer: Recruiting Team (hiring@bjakcareer.com)
  📧 Verifying inbox for 'laura-ann thompson' at 'Air Apps'...
  ✓ Air Apps: laura-ann thompson (lauraann@airapps.com)
  📧 Verifying inbox for 'anastasiia malkina' at 'Mercuryo'...
  ✓ Mercuryo: anastasiia malkina (anastasiia@mercuryo.com)
  📧 Verifying inbox for 'kamil eladas' at 'Hopper'...
  ✓ Hopper: kamil eladas (kamil@hopper.com)
  📧 Verifying inbox for 'cristina ruiz' at 'Claire Joster'...
  ✓ Claire Joster: cristina ruiz (cristina@clairejoster.com)
  📧 Verifying inbox for 'jessica leonardi' at 'Sercanto'...
  ✓ Sercanto: jessica leonardi (jessica@sercanto.com)
  📧 Verifying inbox for 'marie-sophie jammes' at 'Wallapop'...
  ✓ Wallapop: marie-sophie jammes (mariesophie@wallapop.com)
  📧 Verifying inbox for 'ben rodgers' at 'Robert Walters'...
  ✓ Robert Walters: ben rodgers (ben@robertwalters.com)
  📧 Verifying inbox 

### job application agent

agent 1- a RAG agent which has access to my profile pdf, is able to extract relevant information from it like job titles, geo, years of experience, technical capability
Role- Senior Knowdlegdge management Specialist  
Goal- Converting the knowledge into legible, readible and comprehendible structured bullet points without changing any meaning or coherency.  
Backstory- You are a meticulous researcher. You specialize in parsing local documentation
        to answer queries without introducing outside hallucinations.

Task- to process the GIVEN PDF docs without editing anything and storing in memory

(I want to use multiple documents - here resumes)




agent 2 - a job researcher which is able to search for relevant jobs for {candidate_name} specific geo/profile/experience 
domain/industry etc.
Role- A specialist career job finder
Goal- to look for relevant jobs for the profiles provided  as per the user query: {user_query}
Backstory- You have successfully managed to search jobs from all known platforms which may have a job posting  (like LinkedIn/IIm jobs/wellfound, other social media platform posts, public forum posts by individuals and all local job posting sites for each geography provided)

Task- based on user query, find jobs posted online across different job listing platforms across organisations of all sizes posted at any duration
Output- A list of 25 jobs with their job description links 


agent3- a verifier which provides a similarity score b/w my profile doc and job description and background 
and verifies if the score is above a certain threshold
Role- Job Profile Evaluator in a Talent Acquisition Team
Goal- To search for relevancy of profile match 
Backstory- You have a background of matching candidate profiles for organisations ranging from new age startups to big tech and fortune 500 companies across globe. You also check if the job posting is updated, posted by a legitimate company/org

Task- Using the the links given, evaluate which job description title matches the agent 1 document. Cite a brief reason for match in less than 50 words
Output- a score out of Poor/Average/Good/Perfect match for profiles matching my resume. Cite a reason for match in less than 50 words 



agent 4- creates a list of jobs posted in the previous x duration 
Format should be excel with -
Role- Executive Assitant & Data Entry specialist
Goal- You enter data fetched to a legible/readable and structured format like excel for humans to process 
Backstory- You have history of reading and scraping useful information into a human readable structure & format. You are an expert at Excel 

Task- Convert the job matches with Average or higher score in an excel table to be accessed from system
Output-An excel table with the following columns:
1.JOb Profile- scrape the Title of the job
2.Job Link- the web link to directly access the job description
3. Platform Name- fetch/scrape the name of the platform where it is posted
4. Date posted- fetch the date of job post
Append all matched jobs to the above mentioned columns. 

Tools- excel


agent 5 - takes the input as csv and starts finding people on linkedIn who are senior to me or might be hiring in that org, anyone related to tech or management roles near to my role
Role- Networking Specialist
Goal- Headhunting best profiles via using web search
Backstory- You have a track record of hunting down profiles from all social media platforms who have or are likely to hire for relevant roles

Task- Based on the best profile matches, find 3 relevant profiles who are in either management roles or are senior in roles {candidate_name} is looking for
Output- A plan for cold reachout for the mentioned profiles



agent 6 -returns the best profile linkedin URLs + retrieves their email id using appolo as a tool, adds it in another column of the same excel
Role- data entry Specialist  
Goal- You enter data fetched to a legible/readable and structured format like excel for humans to process 
 
Backstory- You have a background in sales navigation and finding ways via official emails/ linkedIn profile links to reach out to prospect employers

Task-To retrieve emails and LinkedIn profiles for people selected by the agent 5
Output- add a column in the given excel with name "Prospect_Employer_Profile_mail" & "Prospect_Employer_Profile_LI" 
A list of 3 LinkedIn profile URLs & their official emails- each in a single cell 
Tools- LinkedIn_Search or google search or apollo.io search (whichever is accessible)




in London, Spain, Netherlands, Singapore, India or any Remote Roles
 